<a href="https://colab.research.google.com/github/zsabro/Batch66/blob/main/HW_task_multiple_agents_and_functions_v02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai-agents -q # Install the open-AI-agents SDK package to import liabraries
!pip install nest_asyncio -q # Install nest_asyncio to fix event loop issue


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 2.4 MB/s eta 0:00:00


In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in notebook environments

from agents import Agent, Runner, AsyncOpenAI, set_default_openai_client, set_tracing_disabled, set_default_openai_api
from google.colab import userdata

# Replace with your actual Gemini API key
gemini_api_key = userdata.get('gemini_api_key')
set_tracing_disabled(True)
set_default_openai_api("chat_completions")

# Set up connection to Gemini AI
external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
set_default_openai_client(external_client)

# Create Triage Agent to decide which agent to call
triage_agent: Agent = Agent(
    name="Triage",
    instructions="You are a triage assistant. Your job is to read the user's prompt and decide which agent to call. If the prompt is about math (e.g., calculations, numbers), call the Math_Agent. If it’s about history (e.g., events, historical figures), call the History_Agent. If the prompt is irrelevant (not about math or history), respond with: 'Sorry, but I can only answer questions about math or history.'",
    model="gemini-2.0-flash"
)

# Create Math Agent for math-related questions
math_agent: Agent = Agent(
    name="Math_Agent",
    instructions="You are a math expert. Answer any math-related questions, such as calculations or explaining math concepts, in a clear and simple way.",
    model="gemini-2.0-flash"
)

# Create History Agent for history-related questions
history_agent: Agent = Agent(
    name="History_Agent",
    instructions="You are a history expert. Answer any history-related questions, such as events or historical figures, in a clear and simple way.",
    model="gemini-2.0-flash"
)

# Function to process the prompt through Triage Agent
def process_prompt(user_prompt):
    # First, run the Triage Agent to decide what to do
    triage_result = Runner.run_sync(triage_agent, user_prompt)
    triage_response = triage_result.final_output.lower()

    # Check Triage Agent's response to decide next steps
    if "math" in triage_response:
        # Call Math Agent
        result = Runner.run_sync(math_agent, user_prompt)
        return result.final_output
    elif "history" in triage_response:
        # Call History Agent
        result = Runner.run_sync(history_agent, user_prompt)
        return result.final_output
    else:
        # Return Triage Agent's response for irrelevant prompts
        return triage_result.final_output



In [ ]:
# Test the system with different prompts
test_prompts = [
    "Calculate 5 + 3",
    "Who was Cleopatra?",
    "What's the weather like?"
]

for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    print(f"Response: {process_prompt(prompt)}\n")

Prompt: Calculate 5 + 3
Response: 5 + 3 = 8


Prompt: Who was Cleopatra?
Response: Cleopatra was the last active ruler of the Ptolemaic Kingdom of Egypt. She was a fascinating figure known for her intelligence, political skills, and romantic relationships with powerful Roman leaders.

Here's a simple breakdown:

*   **Queen of Egypt:** She ruled Egypt from 51 to 30 BC.
*   **Ptolemaic Dynasty:** Cleopatra was part of the Ptolemaic dynasty, which was of Greek origin and had ruled Egypt since the time of Alexander the Great.
*   **Diplomat and Politician:** She was a skilled diplomat and politician who used her intelligence and charm to maintain her power and protect Egypt's interests in the face of Roman expansion.
*   **Relationships with Roman Leaders:** She had famous relationships with Julius Caesar and later with Mark Antony, both of whom were key figures in Roman politics. These relationships were strategic alliances as well as romantic.
*   **Death:** After being defeated by Octa